In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M01_F10_K004_16.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F04_K004_8.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F10_K004_2.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N09_M07_F10_K004_2.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N09_M07_F10_K004_7.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M01_F10_K004_8.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F10_K004_12.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M01_F10_K004_20.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M01_F10_K004_1.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N09_M07_F10_K004_3.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N09_M07_F10_K004_8.mat
/kaggle/input/datasets/dippatel03/paderborn-db/K004/K004.pdf
/kaggle/input/datasets/dippatel03/paderborn-db/K004/N15_M07_F04_K004_9.mat
/kaggle/input/datasets/dippatel03/pa

In [2]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm.notebook import tqdm
from scipy.io import loadmat

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

2026-03-20 06:59:06.056357: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773989946.211428      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773989946.256690      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773989946.619847      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773989946.619885      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773989946.619887      55 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
#Configuration

INPUT_PATH   = '/kaggle/input/datasets/dippatel03/paderborn-db'
WINDOW_SIZE  = 1024   # samples per segment (1024 @ 64kHz ≈ 16 ms)
STRIDE       = 512    # 50% overlap → doubles dataset size
FILES_PER_FOLDER = 20 # default 

BATCH_SIZE   = 64
EPOCHS       = 100    # EarlyStopping will cut this short
LEARNING_RATE = 1e-3

USE_RESIDUAL = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Configuration set.")

Configuration set.


In [4]:
# def check_size(base_path, files_per_folder=5):
#     folders = ['K001']
#     sizes = []

#     print(f"Found {len(folders)} class folders: {folders}\n")

#     for folder in folders:
#         folder_path = os.path.join(base_path, folder)
#         files = sorted(glob.glob(os.path.join(folder_path, "*.mat")))

#         for file_path in files[:files_per_folder]:
#             print(f"\n📂 File: {file_path}")

#             mat_data = loadmat(file_path)

#             for key in mat_data.keys():
#                 if key.startswith("__"):
#                     continue

#                 try:
#                     data = mat_data[key][0, 0]['Y'][0, 0]['Data']
#                     data_arr = np.array(data).flatten().astype(np.float32)

#                     print(f"✅ Key: {key} → shape: {data_arr.shape}")
#                     sizes.append(data_arr.shape)

#                 except Exception as e:
#                     print(f"❌ Key: {key} → Error: {e}")

#     return sizes

# size = check_size(INPUT_PATH)
# print(size)

In [5]:
def extract_paderborn_signal(mat_content):
    """
    Extracts the vibration signal from Paderborn .mat files.
    Structure: mat['FileName'][0,0]['Y'][0,0]['Data']
    Falls back to largest array if structure differs.
    """
    for key in mat_content.keys():
        if key.startswith('__'):
            continue
        try:
            data = mat_content[key][0, 0]['Y'][0, 0]['Data']
            return np.array(data).flatten().astype(np.float32)
        except (IndexError, KeyError, TypeError):
            data_array = np.array(mat_content[key])
            return data_array.flatten().astype(np.float32)
    return None


def load_paderborn_dataset(base_path, files_per_folder=20):
    """
    Loads ALL .mat files from every subfolder.
    Each subfolder = one bearing class label.
    """
    all_signals = []
    all_labels  = []

    folders = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    print(f"Found {len(folders)} class folders: {folders}\n")

    for folder in tqdm(folders, desc="Loading classes"):
        folder_path = os.path.join(base_path, folder)
        files = sorted(glob.glob(os.path.join(folder_path, "*.mat")))
        files_per_folder = len(files)
        # print(f"Number of .mat files in {folder} is : {files_per_folder}\n")

        loaded = 0
        for file_path in files[:files_per_folder]:
            try:
                mat_data = loadmat(file_path)
                signal   = extract_paderborn_signal(mat_data)
                if signal is not None and len(signal) > WINDOW_SIZE:
                    all_signals.append(signal)
                    all_labels.append(folder)
                    loaded += 1
            except Exception as e:
                print(f"  [WARN] {os.path.basename(file_path)}: {e}")

        # print(f"  {folder}: loaded {loaded}/{len(files[:files_per_folder])} files")

    return pd.DataFrame({'signal': all_signals, 'label': all_labels})


print("Starting data load...")
df = load_paderborn_dataset(INPUT_PATH, files_per_folder=FILES_PER_FOLDER)

print(f"\nTotal files loaded : {len(df)}")
print(f"Classes found      : {sorted(df['label'].unique())}")
print(f"Samples per class  :\n{df['label'].value_counts().sort_index()}")

Starting data load...
Found 32 class folders: ['K001', 'K002', 'K003', 'K004', 'K005', 'K006', 'KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30', 'KB23', 'KB24', 'KB27', 'KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']



Loading classes:   0%|          | 0/32 [00:00<?, ?it/s]

  [WARN] N15_M01_F10_KA08_2.mat: Expecting matrix here

Total files loaded : 2559
Classes found      : ['K001', 'K002', 'K003', 'K004', 'K005', 'K006', 'KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30', 'KB23', 'KB24', 'KB27', 'KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
Samples per class  :
label
K001    80
K002    80
K003    80
K004    80
K005    80
K006    80
KA01    80
KA03    80
KA04    80
KA05    80
KA06    80
KA07    80
KA08    79
KA09    80
KA15    80
KA16    80
KA22    80
KA30    80
KB23    80
KB24    80
KB27    80
KI01    80
KI03    80
KI04    80
KI05    80
KI07    80
KI08    80
KI14    80
KI16    80
KI17    80
KI18    80
KI21    80
Name: count, dtype: int64


In [6]:
print(df['signal'][0].shape)


(16008,)


In [7]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

NUM_CLASSES = len(label_encoder.classes_)
print(f"Number of classes : {NUM_CLASSES}")
print(f"Class mapping     :")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i:2d} → {cls}")

Number of classes : 32
Class mapping     :
   0 → K001
   1 → K002
   2 → K003
   3 → K004
   4 → K005
   5 → K006
   6 → KA01
   7 → KA03
   8 → KA04
   9 → KA05
  10 → KA06
  11 → KA07
  12 → KA08
  13 → KA09
  14 → KA15
  15 → KA16
  16 → KA22
  17 → KA30
  18 → KB23
  19 → KB24
  20 → KB27
  21 → KI01
  22 → KI03
  23 → KI04
  24 → KI05
  25 → KI07
  26 → KI08
  27 → KI14
  28 → KI16
  29 → KI17
  30 → KI18
  31 → KI21


In [ ]:
#seperate before segmentation


In [13]:
x = df['signal']
y = df['label']

print(x.shape)


(2559,)


In [15]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    x,y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,   # 0.176 × 0.85 ≈ 0.15 of total
    stratify=y_train_val,
    random_state=RANDOM_SEED
)

In [18]:
print(X_train_val.shape)
print(X_train.shape)
print(y_train.shape)

(2175,)
(1792,)
(1792,)


In [19]:
def segment_train_set(X_train):
    X_train_segement = []
    y_train_segment = []

    for 



SyntaxError: incomplete input (1682777394.py, line 2)

In [8]:
def segment_signals(df, window_size=WINDOW_SIZE, stride=STRIDE):
    """
    Splits each long signal into overlapping windows.
    Each segment is Z-score normalised independently.
    Returns:
        X : np.ndarray of shape (N, window_size)
        y : np.ndarray of shape (N,) with integer labels
    """
    X_list = []
    y_list = []

    for _, row in df.iterrows():
        signal = row['signal']
        label  = row['label_encoded']

        for start in range(0, len(signal) - window_size, stride):
            end     = start + window_size
            segment = signal[start:end]

            # Per-segment Z-score normalisation
            mu  = segment.mean()
            std = segment.std() + 1e-8
            segment = (segment - mu) / std

            X_list.append(segment)
            y_list.append(label)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)


print(f"Segmenting with window={WINDOW_SIZE}, stride={STRIDE}...")
X, y = segment_signals(df)

print(f"X shape : {X.shape}   (samples × time-steps)")
print(f"y shape : {y.shape}")
print(f"\nSegments per class:")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {label_encoder.classes_[u]}: {c}")

Segmenting with window=1024, stride=512...
X shape : (76777, 1024)   (samples × time-steps)
y shape : (76777,)

Segments per class:
  K001: 2400
  K002: 2400
  K003: 2400
  K004: 2400
  K005: 2400
  K006: 2400
  KA01: 2400
  KA03: 2400
  KA04: 2400
  KA05: 2400
  KA06: 2400
  KA07: 2400
  KA08: 2370
  KA09: 2400
  KA15: 2400
  KA16: 2400
  KA22: 2400
  KA30: 2400
  KB23: 2400
  KB24: 2400
  KB27: 2400
  KI01: 2400
  KI03: 2400
  KI04: 2400
  KI05: 2400
  KI07: 2400
  KI08: 2400
  KI14: 2400
  KI16: 2407
  KI17: 2400
  KI18: 2400
  KI21: 2400


In [9]:
# Reshape for Keras: (N, time_steps, channels)
X = X.reshape(X.shape[0], X.shape[1], 1)

# 70% train  |  15% val  |  15% test


print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

# One-hot encode labels for Keras
y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh   = to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES)

Train : (53774, 1024, 1)
Val   : (11486, 1024, 1)
Test  : (11517, 1024, 1)


In [10]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
class_weights_dict = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights computed:")
for i, w in class_weights_dict.items():
    print(f"  Class {i:2d} ({label_encoder.classes_[i]}): {w:.3f}")

Class weights computed:
  Class  0 (K001): 1.000
  Class  1 (K002): 1.000
  Class  2 (K003): 1.000
  Class  3 (K004): 1.000
  Class  4 (K005): 1.000
  Class  5 (K006): 1.000
  Class  6 (KA01): 1.000
  Class  7 (KA03): 1.000
  Class  8 (KA04): 1.000
  Class  9 (KA05): 1.000
  Class 10 (KA06): 1.000
  Class 11 (KA07): 1.000
  Class 12 (KA08): 1.013
  Class 13 (KA09): 1.000
  Class 14 (KA15): 1.000
  Class 15 (KA16): 1.000
  Class 16 (KA22): 1.000
  Class 17 (KA30): 1.000
  Class 18 (KB23): 1.000
  Class 19 (KB24): 1.000
  Class 20 (KB27): 1.000
  Class 21 (KI01): 1.000
  Class 22 (KI03): 1.000
  Class 23 (KI04): 1.000
  Class 24 (KI05): 1.000
  Class 25 (KI07): 1.000
  Class 26 (KI08): 1.000
  Class 27 (KI14): 1.000
  Class 28 (KI16): 0.997
  Class 29 (KI17): 1.000
  Class 30 (KI18): 1.000
  Class 31 (KI21): 1.000


In [11]:
# ──────────────────────────────────────────────────────────────────
# BUILDING BLOCKS
# ──────────────────────────────────────────────────────────────────

def conv_bn_relu(x, filters, kernel_size, strides=1):
    """Conv → BatchNorm → ReLU"""
    x = layers.Conv1D(
        filters, kernel_size,
        strides=strides,
        padding='same',
        use_bias=False,
        kernel_initializer='he_normal'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


# def residual_block(x, filters, kernel_size=3):
#     """
#     [OPTIONAL / ADVANCED]
#     Residual block: two Conv-BN-ReLU + skip connection.
#     If channel dims differ, a 1×1 conv projection is used.
#     """
#     shortcut = x

#     x = conv_bn_relu(x, filters, kernel_size)
#     x = layers.Conv1D(
#         filters, kernel_size,
#         padding='same',
#         use_bias=False,
#         kernel_initializer='he_normal'
#     )(x)
#     x = layers.BatchNormalization()(x)

    # Project shortcut if channel dims differ
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters, 1,
            padding='same',
            use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


# ──────────────────────────────────────────────────────────────────
# MODEL FACTORY
# ──────────────────────────────────────────────────────────────────

def build_cnn(input_shape, num_classes, use_residual=False):
    """
    Deep 1D CNN for vibration-based bearing fault classification.

    Args:
        input_shape  : (window_size, channels) — e.g. (1024, 1)
        num_classes  : number of fault/health categories
        use_residual : if True, uses ResNet-style skip connections
    """
    inputs = Input(shape=input_shape, name='vibration_input')
    x = inputs

    if use_residual:
        # ── RESIDUAL VARIANT ────────────────────────────────────────
        # Block 1
        x = conv_bn_relu(x, 64, kernel_size=7)          # initial broad receptive field
        x = layers.MaxPooling1D(pool_size=4)(x)
        # x = residual_block(x, 64,  kernel_size=5)
        x = layers.MaxPooling1D(pool_size=4)(x)

        # Block 2
        x = residual_block(x, 128, kernel_size=3)
        x = residual_block(x, 128, kernel_size=3)
        x = layers.MaxPooling1D(pool_size=4)(x)

        # Block 3
        x = residual_block(x, 256, kernel_size=3)
        x = residual_block(x, 256, kernel_size=3)

    else:
        # ── STANDARD VARIANT ────────────────────────────────────────
        # Block 1: wide kernel to capture low-freq patterns
        x = conv_bn_relu(x, 64,  kernel_size=7)
        x = layers.MaxPooling1D(pool_size=4)(x)          # 1024 → 256

        # Block 2
        x = conv_bn_relu(x, 128, kernel_size=5)
        x = layers.MaxPooling1D(pool_size=4)(x)          # 256 → 64

        # Block 3
        x = conv_bn_relu(x, 256, kernel_size=3)
        x = layers.MaxPooling1D(pool_size=4)(x)          # 64 → 16

        # Block 4: no downsampling — refine features
        x = conv_bn_relu(x, 256, kernel_size=3)

    # ── HEAD ────────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling1D()(x)               # fixed-size vector, no params

    x = layers.Dense(256, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs,
                  name='ResidualCNN1D' if use_residual else 'DeepCNN1D')
    return model


# Build the model
model = build_cnn(
    input_shape=(WINDOW_SIZE, 1),
    num_classes=NUM_CLASSES,
    use_residual=USE_RESIDUAL
)

model.summary()

I0000 00:00:1773990100.741505      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773990100.747397      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "DeepCNN1D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vibration_input (InputLayer)    │ (None, 1024, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1024, 64)       │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1024, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 128)       │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 64, 256)        │        98,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 16, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 16, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,76

 Total params: 443,104 (1.69 MB)

 Trainable params: 440,928 (1.68 MB)

 Non-trainable params: 2,176 (8.50 KB)

In [12]:
# ── Compile ──────────────────────────────────────────────────────
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

# ── Train ─────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val_oh),
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


I0000 00:00:1773990105.813178     126 service.cc:152] XLA service 0x7957f8015250 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773990105.813223     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773990105.813229     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773990106.434793     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-20 07:01:48.234741: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-20 07:01:48.515675: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 16/841 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.0728 - loss: 3.6617 

I0000 00:00:1773990112.062870     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


841/841 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5216 - loss: 1.6111
Epoch 1: val_accuracy improved from -inf to 0.82161, saving model to best_model.keras
841/841 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.5218 - loss: 1.6102 - val_accuracy: 0.8216 - val_loss: 0.5561 - learning_rate: 0.0010
Epoch 2/100
840/841 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9129 - loss: 0.2574
Epoch 2: val_accuracy improved from 0.82161 to 0.90937, saving model to best_model.keras
841/841 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9129 - loss: 0.2573 - val_accuracy: 0.9094 - val_loss: 0.2481 - learning_rate: 0.0010
Epoch 3/100
839/841 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9517 - loss: 0.1458
Epoch 3: val_accuracy did not improve from 0.90937
841/841 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9517 - loss: 0.1458 - val_accuracy: 0.6424 - val_loss: 1.6588 - learning_rate: 0.0010
Epoch 4/100
841/841 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9677 - loss: 0.0985
Epoch 4: val

KeyboardInterrupt: 